# Transformers, TP 3 : normaliser puis transformer chaque token

Deux briques du bloc Transformer : une LayerNorm écrite à la main (on vérifie sur quel axe elle travaille), puis le MLP appliqué à chaque token séparément, et enfin la connexion résiduelle qui relie les deux.

À rendre : une LayerNorm validée contre nn.LayerNorm, un MLP C → 4C → C et un test de la connexion résiduelle.

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

## 1. Écrire LayerNorm

$$\hat{x}_{i,j} = \frac{x_{i,j} - \mu_i}{\sqrt{\sigma_i^2 + \varepsilon}}, \qquad y_{i,j} = \gamma_j\,\hat{x}_{i,j} + \beta_j$$

Ici i repère le token et j une composante de son embedding : la moyenne et la variance se calculent sur les composantes d'un même token.

In [2]:
class LayerNorm1d:
    """LayerNorm « à la main » : chaque vecteur est normalisé sur sa dernière dimension."""

    def __init__(self, dim, eps=1e-5):
        self.eps = eps
        self.gamma = torch.ones(dim)    # échelle apprise
        self.beta = torch.zeros(dim)    # décalage appris

    def __call__(self, x):
        mu = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_hat = (x - mu) / torch.sqrt(var + self.eps)
        return self.gamma * x_hat + self.beta

### Questions

*Q1. Sur quel axe normaliser ?*

Le dernier (dim=-1), celui des features. Chaque token est normalisé avec ses propres composantes, sans regarder les autres tokens ni le reste du batch. C'est la différence avec BatchNorm.

*Q2. À quoi servent ε, γ et β ?*

ε évite une division par zéro quand la variance d'un vecteur est nulle ou presque. γ et β sont une échelle et un décalage appris pour chaque feature : sans eux, tous les tokens seraient bloqués à moyenne 0 et écart-type 1. Avec eux, le réseau peut corriger, voire annuler, la normalisation si cela l'aide. Ils démarrent à 1 et 0, ce qui donne au départ une normalisation pure.

## 2. Vérifier l'axe

Avant d'exécuter, une prédiction : sur les deux lignes affichées, laquelle donnera environ (0, 1) ? On garde l'entrée x intacte et on range le résultat dans y pour pouvoir comparer ensuite avec nn.LayerNorm sur la même entrée.

In [3]:
torch.manual_seed(1337)
module = LayerNorm1d(100)

x = torch.randn(32, 100)
y = module(x)

print(f"une feature sur 32 tokens  y[:, 0] : moyenne {y[:, 0].mean().item():+.4f}, écart-type {y[:, 0].std().item():.4f}")
print(f"un token sur 100 features  y[0, :] : moyenne {y[0, :].mean().item():+.4f}, écart-type {y[0, :].std().item():.4f}")

une feature sur 32 tokens  y[:, 0] : moyenne +0.1476, écart-type 0.8847
un token sur 100 features  y[0, :] : moyenne -0.0000, écart-type 1.0050


In [4]:
reference = nn.LayerNorm(100)
print("identique à nn.LayerNorm :", torch.allclose(module(x), reference(x), atol=1e-6))

identique à nn.LayerNorm : True


### Questions

*Prédiction.*

C'est la 2ᵉ ligne, y[0, :] : les 100 features d'un même token, puisque LayerNorm normalise chaque token sur ses features. On obtient une moyenne de l'ordre de 10⁻⁹ et un écart-type de 1,005. Ce n'est pas exactement 1 parce que .std() divise par N − 1 alors que la normalisation divise par N : √(100/99) ≈ 1,005.

La 1ʳᵉ ligne, y[:, 0] (une même feature à travers 32 tokens), n'est pas contrainte : on lit une moyenne de 0,15 et un écart-type de 0,88.

*Validation.*

torch.allclose(module(x), nn.LayerNorm(100)(x), atol=1e-6) renvoie True : notre version est équivalente à celle de PyTorch (γ = 1, β = 0, ε = 1e-5 et variance biaisée des deux côtés).

## 3. Le réseau feed-forward

Deux couches linéaires avec une non-linéarité entre les deux : on élargit de C à expansion × C, puis on revient à C. Le facteur d'expansion est un paramètre (4 par défaut) pour pouvoir tester les variantes plus bas.

In [5]:
class FeedForward(nn.Module):

    def __init__(self, n_embd, expansion=4, activation=nn.ReLU):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, expansion * n_embd),
            activation(),
            nn.Linear(expansion * n_embd, n_embd),
        )

    def forward(self, x):
        return self.net(x)


torch.manual_seed(1337)
ffwd = FeedForward(100)
entree = torch.randn(32, 100)
delta = ffwd(entree)
print("forme de delta :", tuple(delta.shape))

cachees = ffwd.net[1](ffwd.net[0](entree))     # activations après ReLU
print(f"part d'activations > 0 : {(cachees > 0).float().mean().item():.1%}")

forme de delta : (32, 100)
part d'activations > 0 : 50.7%


### Questions

*Q1. Quelle proportion d'activations est positive ?*

Environ la moitié (50,7 % ici). Avec des entrées et des poids aléatoires centrés, les pré-activations sont symétriques autour de 0, et ReLU met à zéro à peu près la moitié d'entre elles.

*Q2. Pourquoi revenir à C ?*

La sortie delta est ajoutée à l'entrée par la connexion résiduelle (x + delta), et le bloc suivant attend des vecteurs de taille C. On ne peut additionner que des tenseurs de même forme. Le passage par 4C donne seulement plus de place au calcul, le temps de la couche.

## 4. Ajouter le résiduel (pré-norm)

Organisation pré-norm : LayerNorm d'abord, puis la sous-couche, puis l'addition avec l'entrée. L'énoncé ne fixe pas B, T et n_embd, on prend B = 4, T = 8 et n_embd = 32.

In [6]:
B, T, n_embd = 4, 8, 32

torch.manual_seed(1337)
layer_norm = nn.LayerNorm(n_embd)
ffwd = FeedForward(n_embd)

x = torch.randn(B, T, n_embd)
delta = ffwd(layer_norm(x))
x_out = x + delta

for nom, t in (("x", x), ("delta", delta), ("x + delta", x_out)):
    print(f"{nom:<10} {tuple(t.shape)}")

x          (4, 8, 32)
delta      (4, 8, 32)
x + delta  (4, 8, 32)


## 5. Variantes

On teste GELU à la place de ReLU, plusieurs facteurs d'expansion avec leur nombre de paramètres, et la sortie sans addition résiduelle.

In [7]:
def n_params(module):
    return sum(p.numel() for p in module.parameters())


x_norm = layer_norm(x)

gelu = FeedForward(n_embd, activation=nn.GELU)
print("avec GELU, forme de la sortie :", tuple(gelu(x_norm).shape))

print("\nfacteur  paramètres (C = 32)")
for h in (2, 4, 8):
    print(f"C -> {h}C -> C : {n_params(FeedForward(n_embd, expansion=h)):>6}")

sans_residuel = ffwd(x_norm)
print("\nsans résiduel, forme :", tuple(sans_residuel.shape))

avec GELU, forme de la sortie : (4, 8, 32)

facteur  paramètres (C = 32)
C -> 2C -> C :   4192
C -> 4C -> C :   8352
C -> 8C -> C :  16672

sans résiduel, forme : (4, 8, 32)


### Variantes : ce qu'on observe

1. GELU au lieu de ReLU : il suffit de changer l'activation, les formes ne bougent pas. GELU est une version lissée de ReLU qui laisse passer une petite partie des valeurs négatives ; c'est celle de GPT.
2. et 3. Pour C → hC → C, le nombre de paramètres vaut (C·hC + hC) + (hC·C + C) = 2hC² + hC + C. Avec C = 32 : 4 192 pour h = 2, 8 352 pour h = 4 et 16 672 pour h = 8. Doubler h double presque le nombre de paramètres.
4. Sans résiduel (x_out = delta), la forme reste (4, 8, 32) et rien ne casse. La différence est ailleurs : la sortie ne contient plus l'entrée, et dans un réseau profond le gradient perd son chemin direct, ce qui rend l'entraînement nettement plus difficile.

### Bilan du TP 3

*Quelle dimension LayerNorm conserve-t-elle ?*

Toutes : la sortie a la forme (B, T, C) de l'entrée. LayerNorm ne fait que recentrer et remettre à l'échelle chaque vecteur de taille C.

*Le MLP mélange-t-il les tokens ?*

Non. nn.Linear agit sur la dernière dimension, donc la même transformation est appliquée à chaque token séparément. Dans un bloc Transformer, seule l'attention fait circuler l'information entre positions.

*À quoi sert la connexion résiduelle ?*

Chaque sous-couche devient une correction x + Δx au lieu d'un remplacement. L'information d'origine est conservée et le gradient traverse directement tous les blocs, ce qui stabilise l'entraînement des réseaux profonds.